# Fase C — validazione su silicio e chiusura mesoscopica

**FACCIATA 2.** Questo notebook *chiama e disegna*, non decide: ogni cella invoca
`phase_c.cli` o `phase_c.plots`, cioè lo stesso codice che esegue `run_phase_c.sh`.

Se una cella cominciasse a contenere logica propria, il cancello di parità
(`./run_phase_c.sh parity`) diventerebbe rosso — ed è esattamente ciò che deve fare.

Deve girare **headless** da kernel pulito:

```bash
./run_phase_c.sh notebook
```

Se non gira così, non è riproducibile — e lo si scopre adesso, non a fine campagna.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import matplotlib
matplotlib.use('Agg')          # headless: nessuna finestra, la figura resta nel notebook
import matplotlib.pyplot as plt

from phase_c import cli, plots, RESULTS
print('stadi disponibili:', cli.STADI)
print('senza scheda     :', cli.SENZA_SCHEDA)

## P1 — il plotone

Non ricalcola: **legge l'artefatto** già prodotto. Un grafico che rifacesse i conti per
conto suo mostrerebbe numeri che non stanno in nessun file, e la figura direbbe una cosa
mentre l'artefatto ne dice un'altra.

In [ ]:
P1 = os.path.join(RESULTS, 'p1.json')

if not os.path.isfile(P1):
    print('artefatto assente: eseguo P1 (circa 15 minuti)')
    cli.run_stage('p1', frontend='notebook')

for r in plots.p1_tabella(P1):
    print('N=%(N)-3d mediana %(mediana).3f  p95 %(p95).3f  max %(max).3f  |  '
          'stabili %(stabili)-7s collisioni %(collisioni)-6s TTC %(TTC_min_s).2f s' % r)

In [ ]:
ax = plots.p1_distribuzione(P1)
plt.tight_layout()
plt.show()

La riga rossa è `head-to-tail = 1`. A destra il plotone **amplifica**.

Al crescere di N la fascia attorno a 1 si svuota e la massa si polarizza: è il motivo per
cui il conteggio degli scenari stabili **non è monotono** in N, e per cui quel conteggio da
solo è una statistica che nasconde il fenomeno.

## C0–C3 — gli stadi su silicio

Sono scritti e collaudati contro il mock. Finché la scheda non è accendibile, il `cli`
**dichiara** che serve invece di restituire numeri dal mock come se fossero misure.

In [ ]:
for s in ('c0', 'c1', 'c2', 'c3'):
    try:
        cli.run_stage(s, frontend='notebook')
    except cli.SchedaAssente as e:
        print('%-3s -> %s' % (s, str(e).split('.')[0]))

## Traiettorie — accelerazione rispetto al percorso

Quando C2 avrà girato sulla scheda, `results/c2.json` conterrà le traiettorie e questa
cella le disegna: gap, velocità (ego contro leader) e accelerazione sullo stesso asse dei
tempi. La linea tratteggiata a zero sul gap è la soglia di collisione; quelle a ±16 m/s²
sull'accelerazione sono il dominio del formato di uscita `sfix13_En8`.

In [ ]:
import json
C2 = os.path.join(RESULTS, 'c2.json')
if os.path.isfile(C2):
    d = json.load(open(C2, encoding='utf-8'))['data']
    primo = sorted(d, key=int)[0]
    plots.accel_vs_traiettoria(d[primo], titolo='C2 - scenario %s' % primo)
    plt.tight_layout(); plt.show()
else:
    print('results/c2.json non c\'e ancora: C2 richiede la scheda (RUNBOOK.md).')

### I cancelli di accensione

Due cose vanno **osservate**, non supposte, prima che qualunque numero conti.

**L'XADC legge davvero.** Una lettura fallita non dà errore: dà `0`, che nella eq. 2-9 di
UG480 fa esattamente −273,15 °C. È il tipo di numero che nessuno guarda, perché «è solo la
temperatura» — finché i dati di potenza non risultano inspiegabili. Quando entrambi i percorsi
(sysfs e MMIO) sono disponibili si confrontano: **uno solo non può contraddirsi.**

**Il reset avviene.** Non esiste un bit di reset software: `started` torna a zero solo
riasserendo `ARESETN`, cioè ri-scaricando il bitstream. Ma lascia una firma osservabile —
`done_lat` passa da 1 a 0 — e `prova_firma_del_reset` la controlla. Senza, uno scenario
partirebbe dallo stato del precedente e la Fase C chiamerebbe «silicio» dei numeri sbagliati.

In [ ]:
from phase_c import xadc

try:
    tj_s, vcc_s = xadc.read_tj_sysfs(), xadc.read_vccint_sysfs()
    print('XADC sysfs : Tj %.2f degC, VCCINT %.4f V' % (tj_s, vcc_s))
    print('plausibile :', xadc.verifica_plausibile(tj_s, vcc_s)['ok'])
    try:
        from pynq import MMIO
        m = MMIO(xadc.XADC_BASE, 0x10)
        print('confronto  :', xadc.confronta_percorsi(tj_s, xadc.read_tj(m)))
    except Exception as e:
        print("percorso MMIO non disponibile (%s): resta il solo sysfs, che da solo "
              "non puo contraddirsi." % type(e).__name__)
except xadc.XadcAssente as e:
    print('XADC assente: %s' % e)

## C3 — la campagna di potenza

Il PL consuma 114 mW dentro 1,5–2,5 W di scheda: in assoluto è invisibile a un multimetro da
9999 conteggi. Ma il numero cercato **non è assoluto** — cambia solo il bitstream, e il consumo
del processore, dei regolatori e della periferia si cancella nella differenza.

Cinque condizioni, non tre: `blank/-`, `x1/{on,off}`, `x2/{on,off}`. Su `blank` il PL è vuoto e
il bit di gating non comanda nulla. Il sorteggio copre i **punti**, non le sole configurazioni:
il gating si cambia con una scrittura di registro, e visitarlo sempre nello stesso ordine dentro
ogni configurazione ricreerebbe la correlazione con l'istante che il sorteggio esiste per
rompere.

Il **seme è obbligatorio**: senza, la sequenza «sorteggiata» non l'ha scelta nessuno e la
campagna non è ripetibile.

In [ ]:
C3 = os.path.join(RESULTS, 'c3.json')
SEME = 20260803          # ANNOTARLO: e' cio' che rende la campagna ripetibile

if not os.path.isfile(C3):
    try:
        cli.run_stage('c3', frontend='notebook', seed=SEME, repeats=8)
    except cli.SchedaAssente as e:
        print('%s' % e)

if os.path.isfile(C3):
    for r in plots.c3_tabella(C3):
        print('%-26s %+8.2f mW  +-%6.2f  (per istanza, x%d)  ->  %s'
              % (r['confronto'], r['delta_mW'], r['incertezza_mW'],
                 r['n_istanze'], r['esito']))

`NON separabile` **non è un fallimento della misura: è il risultato.** Se la differenza è
dello stesso ordine dell'incertezza, la risposta corretta è che questo strumento non la
distingue — e va scritta così. Inventare un numero dentro il rumore sarebbe il fallimento.

In [ ]:
if os.path.isfile(C3):
    plots.c3_distribuzione(C3)
    plt.tight_layout(); plt.show()
else:
    print("results/c3.json non c'e ancora: C3 richiede la scheda e il multimetro "
          "(RUNBOOK.md, sezione 4).")

Il grafico qui sotto è il **controllo del sorteggio**, non un risultato. Se i punti di una
condizione si raggruppassero in temperatura, la differenza fra le condizioni conterrebbe anche
la deriva termica — ed è esattamente ciò che l'ordine sorteggiato esiste per impedire. I punti
devono apparire mescolati lungo l'asse orizzontale.

In [ ]:
if os.path.isfile(C3):
    plots.c3_deriva_termica(C3)
    plt.tight_layout(); plt.show()

## Parità fra le due facciate

Lo stesso stadio, eseguito dallo script e dal notebook, deve produrre artefatti identici a
meno dei campi volatili (orario, nome della facciata, directory). La sorgente e la firma del
bitstream **non** sono volatili: un numero prodotto col mock e uno prodotto sul silicio non
sono lo stesso risultato, nemmeno quando coincidono.

```bash
./run_phase_c.sh parity
```